In [1]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import StratifiedKFold, train_test_split
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, GlobalAveragePooling2D, Dense, Dropout, BatchNormalization, MaxPooling2D, Reshape, Multiply
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.applications import VGG16, VGG19, DenseNet121, ResNet50, MobileNet, InceptionV3, Xception
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

2026-11-06 17:24:02.144077: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1793985842.265461      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1793985842.300253      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1793985842.590741      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1793985842.590782      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1793985842.590785      23 computation_placer.cc:177] computation placer alr

In [2]:
# Parameters
input_shape = (224, 224, 3)
num_classes = 5
lr = 1e-5
bs = 32
ep = 30

# Define paths
base_dir = "/kaggle/input/datasets/prahladmehandiratta/cervical-cancer-largest-dataset-sipakmed"
cropped_dirs = {
    cls: os.path.join(base_dir, cls, cls, "CROPPED")
    for cls in ["im_Dyskeratotic", "im_Metaplastic", "im_Koilocytotic", "im_Parabasal", "im_Superficial-Intermediate"]
}
uncropped_dirs = {
    cls: os.path.join(base_dir, cls, cls)
    for cls in ["im_Dyskeratotic", "im_Metaplastic", "im_Koilocytotic", "im_Parabasal", "im_Superficial-Intermediate"]
}

In [3]:
import re
from sklearn.model_selection import StratifiedGroupKFold

def load_images_from_directory(directory, target_size=(224, 224)):
    images, filenames = [], []
    for filename in sorted(os.listdir(directory)):
        img = cv2.imread(os.path.join(directory, filename))
        if img is not None:
            img = cv2.resize(img, target_size)
            images.append(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            filenames.append(filename)
    return np.array(images), filenames

cropped_images, cropped_labels, cropped_groups = {}, {}, {}
for cls, dir_path in cropped_dirs.items():
    images, filenames = load_images_from_directory(dir_path)
    cropped_images[cls] = images
    cropped_labels[cls] = np.full(len(images), cls)
    cropped_groups[cls] = np.array([f"{cls}_{re.split(r'[_.]', fn)[0]}" for fn in filenames])

X_cropped = np.concatenate(list(cropped_images.values()))
y_cropped = np.concatenate(list(cropped_labels.values()))
groups    = np.concatenate(list(cropped_groups.values()))

label_to_index = {cls: idx for idx, cls in enumerate(cropped_dirs.keys())}
y_cropped = np.array([label_to_index[label] for label in y_cropped])
y_cropped = to_categorical(y_cropped, num_classes=5)

def group_split_per_class(y_int, groups, fractions=(0.80, 0.15, 0.05)):
    rng = np.random.RandomState(42)
    split_of_group = {}
    for c in np.unique(y_int):
        idx = np.where(y_int == c)[0]
        ugroups = np.unique(groups[idx])
        rng.shuffle(ugroups)
        sizes = {g: np.sum(groups[idx] == g) for g in ugroups}
        targets = [f * len(idx) for f in fractions]
        filled = [0, 0, 0]
        for g in ugroups:
            s = int(np.argmax([(targets[k] - filled[k]) / targets[k] for k in range(3)]))
            split_of_group[g] = s
            filled[s] += sizes[g]
    return np.array([split_of_group[g] for g in groups])

split_id = group_split_per_class(y_cropped.argmax(axis=1), groups)

X_train, y_train = X_cropped[split_id == 0], y_cropped[split_id == 0]
X_val,   y_val   = X_cropped[split_id == 1], y_cropped[split_id == 1]
X_test,  y_test  = X_cropped[split_id == 2], y_cropped[split_id == 2]

g_train, g_val, g_test = (set(groups[split_id == k]) for k in range(3))
assert not (g_train & g_val) and not (g_train & g_test) and not (g_val & g_test), "DATA LEAK!"
print(f"Train: {len(X_train)} imgs / {len(g_train)} groups")
print(f"Val  : {len(X_val)} imgs / {len(g_val)} groups")
print(f"Test : {len(X_test)} imgs / {len(g_test)} groups")

# ---------- Pool train + val, then build group-aware CV folds once ----------
X_train_temp = np.concatenate([X_train, X_val])
y_train_temp = np.concatenate([y_train, y_val])
groups_temp  = np.concatenate([groups[split_id == 0], groups[split_id == 1]])
y_train_temp_indices = np.argmax(y_train_temp, axis=1)

n_splits = 5
skf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=42)
folds = list(skf.split(X_train_temp, y_train_temp_indices, groups=groups_temp))

for i, (tr, va) in enumerate(folds):
    assert not (set(groups_temp[tr]) & set(groups_temp[va])), f"FOLD {i} LEAK!"
print(f"CV pool: {len(X_train_temp)} imgs | folds ready: {n_splits}")

Train: 3192 imgs / 758 groups
Val  : 630 imgs / 148 groups
Test : 227 imgs / 60 groups
CV pool: 3822 imgs | folds ready: 5


In [4]:
# Data augmentation
datagen = ImageDataGenerator(
    rotation_range=20,         # Random rotation up to 20 degrees
    zoom_range=[0.8, 1.0],     # Zoom out by up to 20%
    brightness_range=[0.9, 1.1],  
    horizontal_flip=True,
    vertical_flip=True,
    preprocessing_function=lambda x: tf.image.adjust_contrast(x, np.random.uniform(0.9, 1.1)),  # Custom contrast adjustment
    fill_mode='nearest'
)

In [5]:
class ASCFBlock(tf.keras.layers.Layer):
    def __init__(self, num_clusters, **kwargs):
        super(ASCFBlock, self).__init__(**kwargs)
        self.num_clusters = num_clusters
        self.conv1x1 = tf.keras.layers.Conv2D(1, kernel_size=1, use_bias=False)
        self.dense = tf.keras.layers.Dense(num_clusters, activation='sigmoid')

    def call(self, feature_maps):
        batch_size = tf.shape(feature_maps)[0]
        h, w, num_channels = feature_maps.shape[1:]
        maps_per_cluster = num_channels // self.num_clusters
        
        reshaped = tf.reshape(feature_maps, (batch_size, h, w, self.num_clusters, maps_per_cluster))
        temp_batch = batch_size * self.num_clusters
        conv_input = tf.reshape(reshaped, (temp_batch, h, w, maps_per_cluster))
        
        fused = self.conv1x1(conv_input)
        fused = tf.reshape(fused, (batch_size, h, w, self.num_clusters))
        
        squeezed = tf.reduce_mean(fused, axis=[1, 2])
        weights = self.dense(squeezed)
        weighted = fused * tf.reshape(weights, (batch_size, 1, 1, self.num_clusters))
        
        final_map = tf.reduce_max(weighted, axis=-1, keepdims=True)
        vector_output = tf.reduce_mean(weighted, axis=[1, 2])
        
        return final_map, vector_output

    def compute_output_spec(self, input_spec):
        batch_size, h, w, num_channels = input_spec.shape
        if num_channels % self.num_clusters != 0:
            raise ValueError(f"Number of channels ({num_channels}) must be divisible by num_clusters ({self.num_clusters})")
        
        final_map_spec = tf.keras.KerasTensor(shape=(batch_size, h, w, 1), dtype=tf.float32)
        vector_output_spec = tf.keras.KerasTensor(shape=(batch_size, self.num_clusters), dtype=tf.float32)
        return final_map_spec, vector_output_spec



def se_block(x, ratio=16):
    channels = x.shape[-1]
    se = GlobalAveragePooling2D()(x)
    se = Dense(channels // ratio, activation='relu')(se)
    se = Dense(channels, activation='sigmoid')(se)
    se = Reshape((1, 1, channels))(se)
    return Multiply()([x, se])

In [6]:
def build_model():
    tf.keras.backend.clear_session()

    input_layer = Input(shape=input_shape)
    
    # Load pretrained ResNet50 base model (without top layers)
    base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    
    # Pass the input_layer to the base model
    x = base_model(input_layer)
    
    x = Conv2D(1024, (3, 3), padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D((1, 1))(x)
    
    x = Conv2D(2048, (3, 3), padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = se_block(x)
    x = MaxPooling2D((1, 1))(x)
    
    # Global Average Pooling
    # x = GlobalAveragePooling2D()(x)
    ascf_layer = ASCFBlock(num_clusters=16)  # Set visualize=True to see feature maps during training
    _, x = ascf_layer(x)
    
    # Additional Dense Layers
    x = Dense(1024, activation='relu')(x)
    x = Dropout(0.5)(x)
    x = Dense(512, activation='relu')(x)
    x = Dropout(0.5)(x)
    
    # Output Layer
    output_layer = Dense(num_classes, activation='softmax')(x)
    
    # Define and compile the model
    model = Model(inputs=input_layer, outputs=output_layer)
    model.compile(optimizer=Adam(learning_rate=lr), loss='categorical_crossentropy', metrics=['accuracy'])
    model.summary()

    return model

# **GAP**

In [7]:
def build_gap_model():
    tf.keras.backend.clear_session()

    input_layer = Input(shape=input_shape)
    
    # Load pretrained ResNet50 base model (without top layers)
    base_model_gap = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    
    # Pass the input_layer to the base model
    x = base_model_gap(input_layer)
    
    x = Conv2D(1024, (3, 3), padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D((1, 1))(x)
    
    x = Conv2D(2048, (3, 3), padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = se_block(x)
    x = MaxPooling2D((1, 1))(x)
    
    # Global Average Pooling
    x = GlobalAveragePooling2D()(x)
    # ascf_layer = ASCFBlock(num_clusters=16)  # Set visualize=True to see feature maps during training
    # _, x = ascf_layer(x)
    
    # Additional Dense Layers
    x = Dense(1024, activation='relu')(x)
    x = Dropout(0.5)(x)
    x = Dense(512, activation='relu')(x)
    x = Dropout(0.5)(x)
    
    # Output Layer
    output_layer_gap = Dense(num_classes, activation='softmax')(x)
    
    # Define and compile the model
    model_gap = Model(inputs=input_layer, outputs=output_layer_gap)
    model_gap.compile(optimizer=Adam(learning_rate=lr), loss='categorical_crossentropy', metrics=['accuracy'])
    model_gap.summary()

    return model_gap

In [8]:
FOLD = 1   

train_idx, val_idx = folds[FOLD]

X_train_fold = X_train_temp[train_idx]
y_train_fold = y_train_temp[train_idx]
X_val_fold   = X_train_temp[val_idx]
y_val_fold   = y_train_temp[val_idx]

print(f"Fold {FOLD+1}/{n_splits}: train {len(X_train_fold)} | val {len(X_val_fold)} | test {len(X_test)}")

datagen.fit(X_train_fold)

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(num_classes),
    y=np.argmax(y_train_fold, axis=1)
)
class_weight_dict = dict(enumerate(class_weights))
print("Class counts (train):", np.sum(y_train_fold, axis=0).astype(int))
print("Class weights:", class_weight_dict)

y_test_true = np.argmax(y_test, axis=1)

# ==================== Main Model ====================
model = build_model()
print("\nTraining Main Model...")
history = model.fit(
    datagen.flow(X_train_fold, y_train_fold, batch_size=bs),
    steps_per_epoch=len(X_train_fold) // bs,
    validation_data=(X_val_fold, y_val_fold),
    epochs=ep,
    verbose=1,
    class_weight=class_weight_dict
)

val_loss, val_acc = model.evaluate(X_val_fold, y_val_fold, verbose=0)
test_loss, _ = model.evaluate(X_test, y_test, verbose=0)
y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)

print(f"\nFold {FOLD+1} - Main Model | Val Accuracy: {val_acc:.4f} | Val Loss: {val_loss:.4f}")
print("Main Model Test Metrics")
print(f"Accuracy : {accuracy_score(y_test_true, y_pred):.5f}")
print(f"Precision: {precision_score(y_test_true, y_pred, average='weighted', zero_division=0):.5f}")
print(f"Recall   : {recall_score(y_test_true, y_pred, average='weighted', zero_division=0):.5f}")
print(f"F1 Score : {f1_score(y_test_true, y_pred, average='weighted', zero_division=0):.5f}")
print(f"Loss     : {test_loss:.5f}")

# ==================== GAP Model ====================
model_gap = build_gap_model()
print("\nTraining GAP Model...")
history_gap = model_gap.fit(
    datagen.flow(X_train_fold, y_train_fold, batch_size=bs),
    steps_per_epoch=len(X_train_fold) // bs,
    validation_data=(X_val_fold, y_val_fold),
    epochs=ep,
    verbose=1,
    class_weight=class_weight_dict
)

val_loss_gap, val_acc_gap = model_gap.evaluate(X_val_fold, y_val_fold, verbose=0)
test_loss_gap, _ = model_gap.evaluate(X_test, y_test, verbose=0)
y_pred_gap = np.argmax(model_gap.predict(X_test, verbose=0), axis=1)

print(f"\nFold {FOLD+1} - GAP Model | Val Accuracy: {val_acc_gap:.4f} | Val Loss: {val_loss_gap:.4f}")
print("GAP Model Test Metrics")
print(f"Accuracy : {accuracy_score(y_test_true, y_pred_gap):.5f}")
print(f"Precision: {precision_score(y_test_true, y_pred_gap, average='weighted', zero_division=0):.5f}")
print(f"Recall   : {recall_score(y_test_true, y_pred_gap, average='weighted', zero_division=0):.5f}")
print(f"F1 Score : {f1_score(y_test_true, y_pred_gap, average='weighted', zero_division=0):.5f}")
print(f"Loss     : {test_loss_gap:.5f}")

Fold 2/5: train 3056 | val 766 | test 227
Class counts (train): [575 586 631 622 642]
Class weights: {0: np.float64(1.0629565217391304), 1: np.float64(1.0430034129692833), 2: np.float64(0.968621236133122), 3: np.float64(0.9826366559485531), 4: np.float64(0.9520249221183801)}


I0000 00:00:1793986674.245901      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1793986674.252059      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet50            │ (None, 7, 7,      │ 23,587,712 │ input_layer[0][0] │
│ (Functional)        │ 2048)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 7, 7,      │ 18,875,392 │ resnet50[0][0]    │
│                     │ 1024)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 7, 7,      │      4,096 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 1024)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 7, 7,      │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 1024)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 7, 7,      │ 18,876,416 │ max_pooling2d[0]… │
│                     │ 2048)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 7, 7,      │      8,192 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 2048)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 2048)      │          0 │ batch_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │    262,272 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 2048)      │    264,192 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 1, 1,      │          0 │ dense_1[0][0]     │
│                     │ 2048)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply (Multiply) │ (None, 7, 7,      │          0 │ batch_normalizat… │
│                     │ 2048)             │            │ reshape[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 7, 7,      │          0 │ multiply[0][0]    │
│ (MaxPooling2D)      │ 2048)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ascf_block          │ [(None, 7, 7, 1), │        400 │ max_pooling2d_1[… │
│ (ASCFBlock)         │ (None, 16)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 1024)      │     17,408 │ ascf_block[0][1]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 1024)      │          0 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 512)       │    524,800 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 512)       │          0 │ dense_4[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 5)         │      2,565 │ dropout_1[0][0] 

 Total params: 62,423,445 (238.13 MB)

... (15 outputs left out)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/30
95/95 ━━━━━━━━━━━━━━━━━━━━ 163s 1s/step - accuracy: 0.5245 - loss: 1.2384 - val_accuracy: 0.6345 - val_loss: 1.0241
Epoch 2/30
 1/95 ━━━━━━━━━━━━━━━━━━━━ 35s 381ms/step - accuracy: 0.8750 - loss: 0.4458

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


95/95 ━━━━━━━━━━━━━━━━━━━━ 3s 30ms/step - accuracy: 0.8750 - loss: 0.4458 - val_accuracy: 0.6410 - val_loss: 1.0148
Epoch 3/30
95/95 ━━━━━━━━━━━━━━━━━━━━ 77s 808ms/step - accuracy: 0.8781 - loss: 0.3947 - val_accuracy: 0.8655 - val_loss: 0.4341
Epoch 4/30
95/95 ━━━━━━━━━━━━━━━━━━━━ 3s 31ms/step - accuracy: 1.0000 - loss: 0.0644 - val_accuracy: 0.8655 - val_loss: 0.4309
Epoch 5/30
95/95 ━━━━━━━━━━━━━━━━━━━━ 76s 807ms/step - accuracy: 0.9152 - loss: 0.2597 - val_accuracy: 0.9256 - val_loss: 0.2125
Epoch 6/30
95/95 ━━━━━━━━━━━━━━━━━━━━ 3s 31ms/step - accuracy: 0.9688 - loss: 0.1214 - val_accuracy: 0.9269 - val_loss: 0.2090
Epoch 7/30
95/95 ━━━━━━━━━━━━━━━━━━━━ 76s 803ms/step - accuracy: 0.9214 - loss: 0.2222 - val_accuracy: 0.9595 - val_loss: 0.1214
Epoch 8/30
95/95 ━━━━━━━━━━━━━━━━━━━━ 3s 31ms/step - accuracy: 0.9688 - loss: 0.0841 - val_accuracy: 0.9595 - val_loss: 0.1216
Epoch 9/30
95/95 ━━━━━━━━━━━━━━━━━━━━ 76s 798ms/step - accuracy: 0.9525 - loss: 0.1446 - val_accuracy: 0.9752 - val_

In [9]:
print("\n" + "="*70)
print(f"FOLD {FOLD+1}/{n_splits} - TEST SET RESULTS")
print("="*70)

# ===================== MAIN MODEL =====================
print("\n--- Main Model ---")
print(f"Test Accuracy : {accuracy_score(y_test_true, y_pred):.5f}")
print(f"Test Precision: {precision_score(y_test_true, y_pred, average='weighted', zero_division=0):.5f}")
print(f"Test Recall   : {recall_score(y_test_true, y_pred, average='weighted', zero_division=0):.5f}")
print(f"Test F1 Score : {f1_score(y_test_true, y_pred, average='weighted', zero_division=0):.5f}")
print(f"Test Loss     : {test_loss:.5f}")

# ===================== GAP MODEL =====================
print("\n--- GAP Model ---")
print(f"Test Accuracy : {accuracy_score(y_test_true, y_pred_gap):.5f}")
print(f"Test Precision: {precision_score(y_test_true, y_pred_gap, average='weighted', zero_division=0):.5f}")
print(f"Test Recall   : {recall_score(y_test_true, y_pred_gap, average='weighted', zero_division=0):.5f}")
print(f"Test F1 Score : {f1_score(y_test_true, y_pred_gap, average='weighted', zero_division=0):.5f}")
print(f"Test Loss     : {test_loss_gap:.5f}")


FOLD 2/5 - TEST SET RESULTS

--- Main Model ---
Test Accuracy : 0.96035
Test Precision: 0.96374
Test Recall   : 0.96035
Test F1 Score : 0.96063
Test Loss     : 0.13574

--- GAP Model ---
Test Accuracy : 0.94714
Test Precision: 0.95020
Test Recall   : 0.94714
Test F1 Score : 0.94745
Test Loss     : 0.20442
